# 01 — Построение датасета WordNet

Этот ноутбук строит курированный список слов для RSA-анализа семантических иерархий.

**Источник:** WordNet — лексическая база данных, организованная в виде иерархии синсетов.

**Структура датасета:**
- 5 семантических категорий: animal, food, furniture, tool, vehicle
- 4 уровня иерархии (level 1 = корень домена, level 4 = лист)
- Флаг типичности `is_typical` (по Rosch, 1975) - ручная разметка

**Выходные файлы:**
- `outputs/dataset.json` — 123 слова с метаданными WordNet
- `outputs/dataset_report.txt` — отчёт о фильтрации

## Импорты и настройка среды

In [1]:
import json
import os
import pandas as pd
from nltk.corpus import wordnet as wn

os.makedirs("outputs", exist_ok=True)

## Список кандидатов

Вручную составленный список 145 слов с аннотациями:
- `word` — само слово
- `synset_id` — идентификатор целевого синсета в WordNet
- `level` — уровень в иерархии (1 = наиболее общее, 4 = наиболее конкретное)
- `category` — семантическая категория
- `is_typical` — типичный (True) или атипичный (False) exemplar по Rosch - РУЧНАЯ РАЗМЕТКА

Атипичные примеры: penguin (нелетающая птица), bat (летающее млекопитающее),
tomato (биологически фрукт), skateboard (транспорт без двигателя) и др.

In [ ]:
CANDIDATES = [
    # ── ANIMALS ──────────────────────────────────────────────────────────
    ("animal",      "animal.n.01",      1, "animal", True),
    ("mammal",      "mammal.n.01",      2, "animal", True),
    ("bird",        "bird.n.01",        2, "animal", True),
    ("fish",        "fish.n.01",        2, "animal", True),
    ("reptile",     "reptile.n.01",     2, "animal", True),
    ("dog",         "dog.n.01",         3, "animal", True),
    ("cat",         "cat.n.01",         3, "animal", True),
    ("horse",       "horse.n.01",       3, "animal", True),
    ("rabbit",      "rabbit.n.01",      3, "animal", True),
    ("sparrow",     "sparrow.n.01",     3, "animal", True),   # proto-bird (Rosch)
    ("eagle",       "eagle.n.01",       3, "animal", True),
    ("penguin",     "penguin.n.01",     3, "animal", False),  # atypical bird
    ("ostrich",     "ostrich.n.01",     3, "animal", False),  # atypical bird
    ("salmon",      "salmon.n.01",      3, "animal", True),
    ("shark",       "shark.n.01",       3, "animal", True),
    ("dachshund",   "dachshund.n.01",   4, "animal", True),
    ("poodle",      "poodle.n.01",      4, "animal", True),
    ("siamese",     "siamese_cat.n.01", 4, "animal", True),
    ("mallard",     "mallard.n.01",     4, "animal", True),
    ("canary",      "canary.n.02",      4, "animal", True),
    ("robin",       "robin.n.01",       3, "animal", True),   # proto-bird
    ("crow",        "crow.n.01",        3, "animal", True),   # proto-bird
    ("emu",         "emu.n.02",         3, "animal", False),  # atypical bird (flightless)
    ("dalmatian",   "dalmatian.n.02",   4, "animal", False),
    ("bat",         "bat.n.01",         3, "animal", False),  # atypical: flying mammal
    ("whale",       "whale.n.02",       3, "animal", False),  # atypical: marine mammal
    ("platypus",    "platypus.n.01",    3, "animal", False),  # atypical: egg-laying mammal

    # ── VEHICLES ─────────────────────────────────────────────────────────
    ("vehicle",     "vehicle.n.01",     1, "vehicle", True),
    ("car",         "car.n.01",         2, "vehicle", True),
    ("aircraft",    "aircraft.n.01",    2, "vehicle", True),
    ("boat",        "boat.n.01",        2, "vehicle", True),
    ("truck",       "truck.n.01",       3, "vehicle", True),
    ("sedan",       "sedan.n.01",       3, "vehicle", True),
    ("jeep",        "jeep.n.01",        3, "vehicle", True),
    ("jet",         "jet.n.01",         3, "vehicle", True),
    ("helicopter",  "helicopter.n.01",  3, "vehicle", True),
    ("canoe",       "canoe.n.01",       3, "vehicle", True),
    ("blimp",       "blimp.n.01",       3, "vehicle", False),
    ("gondola",     "gondola.n.03",     3, "vehicle", False),
    ("limousine",   "limousine.n.01",   4, "vehicle", True),
    ("convertible", "convertible.n.01", 4, "vehicle", True),
    ("biplane",     "biplane.n.01",     4, "vehicle", True),
    ("bus",         "bus.n.01",         3, "vehicle", True),
    ("motorcycle",  "motorcycle.n.01",  3, "vehicle", True),
    ("bicycle",     "bicycle.n.01",     3, "vehicle", True),
    ("train",       "train.n.01",       3, "vehicle", True),
    ("ferry",       "ferry.n.01",       3, "vehicle", True),
    ("yacht",       "yacht.n.01",       4, "vehicle", True),
    ("ambulance",   "ambulance.n.01",   4, "vehicle", True),
    ("rickshaw",    "jinrikisha.n.01",  3, "vehicle", False),
    ("hovercraft",  "hovercraft.n.01",  3, "vehicle", False),
    ("skateboard",  "skateboard.n.01",  3, "vehicle", False),
    ("dogsled",     "dogsled.n.01",     3, "vehicle", False),
    ("glider",      "glider.n.01",      3, "vehicle", False),
    ("raft",        "raft.n.01",        3, "vehicle", False),

    # ── TOOLS ────────────────────────────────────────────────────────────
    ("tool",        "tool.n.01",        1, "tool", True),
    ("hand_tool",   "hand_tool.n.01",   2, "tool", True),
    ("edge_tool",   "edge_tool.n.01",   2, "tool", True),
    ("hammer",      "hammer.n.01",      3, "tool", True),
    ("screwdriver", "screwdriver.n.01", 3, "tool", True),
    ("wrench",      "wrench.n.01",      3, "tool", True),
    ("saw",         "saw.n.02",         3, "tool", True),
    ("chisel",      "chisel.n.01",      3, "tool", True),
    ("pliers",      "pliers.n.01",      3, "tool", True),
    ("mallet",      "mallet.n.01",      4, "tool", True),
    ("sledgehammer","maul.n.01",        4, "tool", False),
    ("hacksaw",     "hacksaw.n.01",     4, "tool", True),
    ("knife",       "knife.n.01",       3, "tool", True),
    ("drill",       "drill.n.01",       3, "tool", True),
    ("awl",         "awl.n.01",         4, "tool", True),
    ("tongs",       "tongs.n.01",       3, "tool", True),
    ("spatula",     "spatula.n.01",     3, "tool", True),
    ("shovel",      "shovel.n.01",      3, "tool", True),
    ("axe",         "ax.n.01",          3, "tool", True),
    ("crowbar",     "crowbar.n.01",     3, "tool", False),
    ("trowel",      "trowel.n.01",      4, "tool", False),
    ("stapler",     "stapler.n.01",     3, "tool", False),
    ("scalpel",     "scalpel.n.01",     3, "tool", False),
    ("forceps",     "forceps.n.01",     3, "tool", False),

    # ── FOOD ─────────────────────────────────────────────────────────────
    ("food",        "food.n.01",        1, "food", True),
    ("fruit",       "fruit.n.01",       2, "food", True),
    ("vegetable",   "vegetable.n.01",   2, "food", True),
    ("grain",       "grain.n.01",       2, "food", True),
    ("apple",       "apple.n.01",       3, "food", True),
    ("banana",      "banana.n.01",      3, "food", True),
    ("orange",      "orange.n.01",      3, "food", True),
    ("tomato",      "tomato.n.01",      3, "food", False),
    ("carrot",      "carrot.n.01",      3, "food", True),
    ("broccoli",    "broccoli.n.01",    3, "food", False),
    ("wheat",       "wheat.n.01",       3, "food", True),
    ("pineapple",   "pineapple.n.01",   4, "food", False),
    ("strawberry",  "strawberry.n.01",  4, "food", True),
    ("lemon",       "lemon.n.01",       4, "food", True),
    ("bread",       "bread.n.01",       3, "food", True),
    ("rice",        "rice.n.01",        3, "food", True),
    ("potato",      "potato.n.01",      3, "food", True),
    ("grape",       "grape.n.01",       3, "food", True),
    ("mango",       "mango.n.01",       4, "food", True),
    ("corn",        "corn.n.01",        3, "food", True),
    ("avocado",     "avocado.n.01",     3, "food", False),
    ("cucumber",    "cucumber.n.02",    3, "food", False),
    ("coconut",     "coconut.n.02",     3, "food", False),

    # ── FURNITURE ────────────────────────────────────────────────────────
    ("furniture",   "furniture.n.01",   1, "furniture", True),
    ("seat",        "seat.n.01",        2, "furniture", True),
    ("table",       "table.n.01",       2, "furniture", True),
    ("bed",         "bed.n.01",         2, "furniture", True),
    ("wardrobe",    "wardrobe.n.01",    2, "furniture", True),
    ("chair",       "chair.n.01",       3, "furniture", True),
    ("sofa",        "sofa.n.01",        3, "furniture", True),
    ("stool",       "stool.n.01",       3, "furniture", True),
    ("desk",        "desk.n.01",        3, "furniture", True),
    ("bench",       "bench.n.01",       3, "furniture", True),
    ("bookcase",    "bookcase.n.01",    3, "furniture", True),
    ("cabinet",     "cabinet.n.01",     3, "furniture", True),
    ("dresser",     "chest_of_drawers.n.01", 3, "furniture", True),
    ("crib",        "crib.n.01",        3, "furniture", True),
    ("ottoman",     "ottoman.n.02",     4, "furniture", False),
    ("armchair",    "armchair.n.01",    4, "furniture", True),
    ("recliner",    "recliner.n.01",    4, "furniture", True),
    ("loveseat",    "love_seat.n.01",   4, "furniture", True),
    ("hammock",     "hammock.n.02",     3, "furniture", False),
    ("futon",       "futon.n.01",       3, "furniture", False),
    ("pew",         "pew.n.01",         3, "furniture", False),
    ("beanbag",     "beanbag.n.01",     3, "furniture", False),
    ("cradle",      "cradle.n.01",      3, "furniture", False),
    ("highchair",   "highchair.n.01",   3, "furniture", False),
]

# Убираем дубликаты (оставляем первое вхождение)
seen = set()
CANDIDATES_DEDUP = []
for c in CANDIDATES:
    if c[0] not in seen:
        seen.add(c[0])
        CANDIDATES_DEDUP.append(c)
CANDIDATES = CANDIDATES_DEDUP

print(f"Кандидатов после дедупликации: {len(CANDIDATES)}")

Кандидатов после дедупликации: 126


## Вспомогательные функции WordNet

Функции для проверки синсетов, подсчёта полисемии и извлечения цепочек гиперонимов.

- `count_synsets(word)` — индекс полисемии: количество именных синсетов
- `target_synset_rank(word, synset_id)` — позиция целевого синсета в списке (rank 0 = первичное значение)
- `get_hypernym_chain(synset_id)` — цепочка от корня до листа
- `verify_synset(synset_id)` — проверка существования синсета

In [3]:
def count_synsets(word: str) -> int:
    """Количество именных синсетов слова (индекс полисемии)."""
    return len(wn.synsets(word, pos=wn.NOUN))


def target_synset_rank(word: str, synset_id: str) -> int:
    """
    Позиция целевого синсета в списке именных синсетов слова.
    Rank 0 = наиболее частотное / первичное значение.
    Возвращает -1, если синсет не найден в списке существительных.
    """
    synsets = wn.synsets(word, pos=wn.NOUN)
    for i, sy in enumerate(synsets):
        if sy.name() == synset_id:
            return i
    return -1


def get_hypernym_chain(synset_id: str, max_depth: int = 8) -> list:
    """Цепочка гиперонимов от корня до синсета (root → leaf)."""
    try:
        sy = wn.synset(synset_id)
    except Exception:
        return []
    chain = [sy.lemmas()[0].name().replace("_", " ")]
    current = sy
    for _ in range(max_depth):
        hypernyms = current.hypernyms()
        if not hypernyms:
            break
        current = hypernyms[0]
        chain.append(current.lemmas()[0].name().replace("_", " "))
    return list(reversed(chain))  # root → leaf


def verify_synset(synset_id: str) -> bool:
    """Проверяет, существует ли синсет в WordNet."""
    try:
        wn.synset(synset_id)
        return True
    except Exception:
        return False

## Двухступенчатый фильтр

Функция `build()` применяет два критерия:

1. **Rank ≤ 1** — целевой синсет должен быть первичным или вторичным значением слова.
   Это исключает омонимы, где нужное значение периферийно (например, "bank" как берег
   при целевом синсете финансового учреждения).

2. **n_synsets ≤ 12** — исключает гиперполисемичные слова.
   Критерий мягче строгой моносемии, но сохраняет конкретные существительные:
   dog (7 синсетов, rank 0 — включается), bank (10+ синсетов — исключается).

In [4]:
def build(max_rank: int = 1, max_synsets: int = 10) -> tuple:
    """
    Применяет двухступенчатый фильтр к списку кандидатов.
    Возвращает (accepted_records, rejected_list).
    """
    records = []
    rejected = []

    for word, synset_id, level, category, is_typical in CANDIDATES:
        if not verify_synset(synset_id):
            rejected.append((word, "synset not found"))
            continue

        n_syn = count_synsets(word)
        rank  = target_synset_rank(word, synset_id)

        if rank == -1:
            rejected.append((word, f"synset {synset_id} not in noun list"))
            continue
        if rank > max_rank:
            rejected.append((word, f"rank={rank} > {max_rank} (not primary sense)"))
            continue
        if n_syn > max_synsets:
            rejected.append((word, f"n_synsets={n_syn} > {max_synsets}"))
            continue

        chain = get_hypernym_chain(synset_id)

        records.append({
            "word":           word,
            "synset_id":      synset_id,
            "level":          level,
            "category":       category,
            "is_typical":     is_typical,
            "n_synsets":      n_syn,
            "synset_rank":    rank,
            "hypernym_chain": chain,
        })

    return records, rejected

## Функция отчёта

Формирует читаемый отчёт о принятых и отклонённых словах по категориям.

In [5]:
def print_report(records: list, rejected: list) -> str:
    lines = []
    lines.append("=" * 60)
    lines.append("DATASET CURATION REPORT  (rank ≤ 1, n_synsets ≤ 12)")
    lines.append("=" * 60)
    lines.append(f"Accepted: {len(records)}   Rejected: {len(rejected)}\n")

    for cat in sorted({r["category"] for r in records}):
        cat_words = [r for r in records if r["category"] == cat]
        lines.append(f"── {cat.upper()} ({len(cat_words)} words) ──")
        for r in sorted(cat_words, key=lambda x: (x["level"], x["word"])):
            typ = "✓" if r["is_typical"] else "✗"
            lines.append(
                f"  L{r['level']} {typ}  {r['word']:<16} "
                f"synsets={r['n_synsets']} rank={r['synset_rank']}  "
                f"chain: {' → '.join(r['hypernym_chain'][-4:])}"
            )
        lines.append("")

    lines.append("── REJECTED ──")
    for word, reason in rejected:
        lines.append(f"  {word:<16} {reason}")

    report = "\n".join(lines)
    print(report)
    return report

## Построение датасета

Запускаем фильтрацию с параметрами `max_rank=1, max_synsets=12`.
Результаты сохраняются в `outputs/dataset.json` и `outputs/dataset_report.txt`.

In [6]:
print("Строим датасет WordNet …")
records, rejected = build(max_rank=1, max_synsets=12)

report = print_report(records, rejected)

Строим датасет WordNet …
DATASET CURATION REPORT  (rank ≤ 1, n_synsets ≤ 12)
Accepted: 123   Rejected: 3

── ANIMAL (26 words) ──
  L1 ✓  animal           synsets=1 rank=0  chain: whole → living thing → organism → animal
  L2 ✓  bird             synsets=5 rank=0  chain: animal → chordate → vertebrate → bird
  L2 ✓  fish             synsets=4 rank=0  chain: chordate → vertebrate → aquatic vertebrate → fish
  L2 ✓  mammal           synsets=1 rank=0  chain: animal → chordate → vertebrate → mammal
  L2 ✓  reptile          synsets=1 rank=0  chain: animal → chordate → vertebrate → reptile
  L3 ✗  bat              synsets=5 rank=0  chain: vertebrate → mammal → placental → bat
  L3 ✓  cat              synsets=8 rank=0  chain: placental → carnivore → feline → cat
  L3 ✓  crow             synsets=6 rank=0  chain: passerine → oscine → corvine bird → crow
  L3 ✓  dog              synsets=7 rank=0  chain: organism → animal → domestic animal → dog
  L3 ✓  eagle            synsets=4 rank=0  chain: ve

In [7]:
# Сохраняем датасет
out_json = "outputs/dataset.json"
with open(out_json, "w") as f:
    json.dump(records, f, indent=2, ensure_ascii=False)

# Сохраняем текстовый отчёт
out_txt = "outputs/dataset_report.txt"
with open(out_txt, "w") as f:
    f.write(report)

print(f"\nСохранено → {out_json}  ({len(records)} слов)")
print(f"Сохранено → {out_txt}")


Сохранено → outputs/dataset.json  (123 слов)
Сохранено → outputs/dataset_report.txt


## Просмотр итогового датасета

Загружаем сохранённый JSON и отображаем как DataFrame для проверки.

In [8]:
df = pd.DataFrame(records)
df["hypernym_chain"] = df["hypernym_chain"].apply(lambda c: " → ".join(c[-4:]))
print(f"Итого слов: {len(df)}  |  Категорий: {df['category'].nunique()}")
print("\nРаспределение по категориям и уровням:")
print(df.groupby(["category", "level"])[["word"]].count().rename(columns={"word": "n_words"}))
df.head(20)

Итого слов: 123  |  Категорий: 5

Распределение по категориям и уровням:
                 n_words
category  level         
animal    1            1
          2            4
          3           16
          4            5
food      1            1
          2            3
          3           15
          4            4
furniture 1            1
          2            4
          3           15
          4            4
tool      1            1
          2            2
          3           16
          4            5
vehicle   1            1
          2            3
          3           17
          4            5


,word,synset_id,level,category,is_typical,n_synsets,synset_rank,hypernym_chain
0,animal,animal.n.01,1,animal,True,1,0,whole → living thing → organism → animal
1,mammal,mammal.n.01,2,animal,True,1,0,animal → chordate → vertebrate → mammal
2,bird,bird.n.01,2,animal,True,5,0,animal → chordate → vertebrate → bird
3,fish,fish.n.01,2,animal,True,4,0,chordate → vertebrate → aquatic vertebrate → fish
4,reptile,reptile.n.01,2,animal,True,1,0,animal → chordate → vertebrate → reptile
5,dog,dog.n.01,3,animal,True,7,0,organism → animal → domestic animal → dog
6,cat,cat.n.01,3,animal,True,8,0,placental → carnivore → feline → cat
7,horse,horse.n.01,3,animal,True,5,0,ungulate → odd-toed ungulate → equine → horse
8,rabbit,rabbit.n.01,3,animal,True,3,0,placental → lagomorph → leporid → rabbit
9,sparrow,sparrow.n.01,3,animal,True,2,0,vertebrate → bird → passerine → sparrow


In [9]:
print("Атипичные exemplars (is_typical=False):")
print(df[~df["is_typical"]][["word", "category", "level"]].to_string(index=False))

Атипичные exemplars (is_typical=False):
        word  category  level
     penguin    animal      3
     ostrich    animal      3
         emu    animal      3
   dalmatian    animal      4
         bat    animal      3
       whale    animal      3
    platypus    animal      3
    rickshaw   vehicle      3
  hovercraft   vehicle      3
  skateboard   vehicle      3
     dogsled   vehicle      3
      glider   vehicle      3
        raft   vehicle      3
sledgehammer      tool      4
     crowbar      tool      3
      trowel      tool      4
     stapler      tool      3
     scalpel      tool      3
     forceps      tool      3
      tomato      food      3
    broccoli      food      3
   pineapple      food      4
     avocado      food      3
    cucumber      food      3
     coconut      food      3
     ottoman furniture      4
     hammock furniture      3
       futon furniture      3
         pew furniture      3
     beanbag furniture      3
      cradle furniture      3
